# IFN680 Project 5 - LLMForward: Addition + Subtraction (left-to-right)

**Group 4** — Karan Rooprai (n12498122), Nhu Hieu Nguyen (n12194778)

This notebook extends the Week 7 addition LLM to also perform **subtraction**
(`a-b=c`, negative results allowed) and trains it in **Forward mode**.
The answer is predicted left-to-right, as in the workshop.
The architecture, training objective and helper structure follow the 7.3 workshop.

In [ ]:
# Core libraries (same stack as the Week 7 workshop)
import torch
from torch import nn
from torch.nn import functional as F
import random, math, re, time, pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
REVERSE = False   # forward (left-to-right) target order

## Step 1 — Tokenizer

In [ ]:
# --- Tokenizer -------------------------------------------------------------
# Vocabulary is the workshop's plus a single '-' token, which serves BOTH as the
# subtraction operator ("50-73=") and as the negative sign in a result ("-23").
# Ids: 0-9 digits, 10 '+', 11 '-', 12 '=', 13 [PAD], 14 [EOS].
pad_token = "[PAD]"
eos_token = "[EOS]"

class character_level_tokenizer:
    def __init__(self):
        self.vocab = [str(x) for x in range(10)] + ["+", "-", "="] + [pad_token, eos_token]
        self.token_to_id = {v: k for k, v in enumerate(self.vocab)}
        self.id_to_token = {k: v for k, v in enumerate(self.vocab)}
        self.ntokens = len(self.vocab)
        self.pattern = f"[^{re.escape(''.join(self.vocab))}]"
    def clean(self, text):
        return re.sub(self.pattern, "", text)
    def pre_tokenization(self, text):
        return [c for c in text]
    def encode(self, text):
        return [self.token_to_id[c] for c in self.pre_tokenization(self.clean(text))]
    def decode(self, token_list):
        return "".join([self.id_to_token[i] for i in token_list])

tokenizer = character_level_tokenizer()
ntokens = tokenizer.ntokens
print("ntokens:", ntokens, "| vocab:", tokenizer.vocab)

## Step 2 — Data (balanced addition + subtraction)

Same generator as the workshop, extended with a `-` operation and a shared, balanced test set that is saved to disk so both models and `main_report.ipynb` evaluate on **identical** examples.

In [ ]:
# --- Data generation: a (+|-) b = c ---------------------------------------
# Operands are non-negative, at most `number_digits` digits. Subtraction may be
# negative (b > a), rendered by str() with a leading '-'.
def sample_datapoint(number_digits=3, operation=None):
    if operation is None:
        operation = random.choice(["+", "-"])
    hi = 10 ** number_digits
    a_int = random.randint(0, hi - 1)
    b_int = random.randint(0, hi - 1)
    result = a_int + b_int if operation == "+" else a_int - b_int
    return f"{a_int}{operation}{b_int}=", str(result)

def build_dataset(train_size, val_size, test_size, number_digits=3, seed=SEED):
    """Unique, non-overlapping splits with a balanced 50/50 add/sub mix."""
    rng = random.Random(seed)
    total = train_size + val_size + test_size
    per_op = total // 2 + 1
    def unique_for(op, n):
        seen, out, hi = set(), [], 10 ** number_digits
        while len(out) < n:
            a, b = rng.randint(0, hi - 1), rng.randint(0, hi - 1)
            prompt = f"{a}{op}{b}="
            if prompt in seen:
                continue
            seen.add(prompt)
            result = a + b if op == "+" else a - b
            out.append((prompt, str(result)))
        return out
    add, sub = unique_for("+", per_op), unique_for("-", per_op)
    mixed = []
    for pa, pb in zip(add, sub):
        mixed += [pa, pb]
    rng.shuffle(mixed)
    return (mixed[:train_size],
            mixed[train_size:train_size + val_size],
            mixed[train_size + val_size:train_size + val_size + test_size])

# --- Reverse-mode transform (literal full-string reversal) ------------------
# "31"->"13", "1998"->"8991", "-123"->"321-" (sign predicted LAST).
def to_reverse(s):   return s[::-1]
def from_reverse(s): return s[::-1]

def parse_answer(answer_str, reverse=False):
    """Decoded answer string -> int (un-reversing first in reverse mode)."""
    s = from_reverse(answer_str) if reverse else answer_str
    neg = s.startswith("-")
    digits = "".join(ch for ch in s if ch.isdigit())
    if digits == "":
        return None
    return -int(digits) if neg else int(digits)

# --- Carry (add) / borrow (sub) structure, units-first ----------------------
def carry_positions(a, b):
    width = max(len(str(a)), len(str(b)), len(str(a + b)))
    sa, sb = str(a).zfill(width), str(b).zfill(width)
    carry, flags = 0, []
    for i in reversed(range(width)):
        flags.append(carry)
        carry = 1 if int(sa[i]) + int(sb[i]) + carry >= 10 else 0
    return flags

def borrow_positions(a, b):
    hi, lo = (a, b) if a >= b else (b, a)
    width = max(len(str(hi)), len(str(lo)))
    sh, sl = str(hi).zfill(width), str(lo).zfill(width)
    borrow, flags = 0, []
    for i in reversed(range(width)):
        top, bot = int(sh[i]) - borrow, int(sl[i])
        flags.append(1 if top < bot else 0)
        borrow = 1 if top < bot else 0
    return flags

def operation_of(prompt):
    return "+" if "+" in prompt else "-"

In [ ]:
# Dataset sizes. Subtraction (borrows, negatives) and especially the Reverse ordering are
# much harder to learn than plain addition: at 80k examples the Reverse model plateaus,
# but at 150k it converges. Both modes use the same 150k/10k/10k split for a fair comparison.
train_size, val_size, test_size = 150000, 10000, 10000
data_train, data_val, data_test = build_dataset(train_size, val_size, test_size)

# Save the shared held-out test set once (used by the other model and main_report).
import os
if not os.path.exists("project5_testset.pkl"):
    with open("project5_testset.pkl", "wb") as f:
        pickle.dump(data_test, f)
    print("saved project5_testset.pkl")
else:  # reuse the canonical test set so both models see the same examples
    with open("project5_testset.pkl", "rb") as f:
        data_test = pickle.load(f)
    print("loaded shared project5_testset.pkl")

n_sub = sum(1 for p, _ in data_test if "-" in p)
print(f"train {len(data_train)}  val {len(data_val)}  test {len(data_test)} "
      f"(test sub fraction {n_sub/len(data_test):.2f})")
print("examples:", data_train[:4])

## Step 3 — Positional encoding

In [ ]:
# --- Positional encoding (sinusoidal, from the workshop) --------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp((torch.arange(0, d_model, 2).float() / d_model) * (-math.log(1e4)))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)
    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

## Step 4 — Transformer model

In [ ]:
# --- Transformer with causal (masked) self-attention ------------------------
class CustomEncoderLayer(nn.TransformerEncoderLayer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_weights = None
    def forward(self, src, src_mask=None, src_key_padding_mask=None, **kwargs):
        src2, attn_weights = self.self_attn(
            src, src, src, attn_mask=src_mask,
            key_padding_mask=src_key_padding_mask,
            need_weights=True, average_attn_weights=False)
        self.attn_weights = attn_weights
        src = src + self.dropout1(src2); src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2); src = self.norm2(src)
        return src

class TransformerModel(nn.Module):
    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):
        super().__init__()
        self.input_emb = nn.Embedding(ntoken, ninp)
        self.pos_encoder = PositionalEncoding(ninp, dropout)
        encoder_layers = CustomEncoderLayer(ninp, nhead, nhid, 0.1)
        self.encoder = nn.TransformerEncoder(encoder_layers, nlayers)
        self.decoder = nn.Linear(ninp, ntoken)
        self.ninp = ninp
        self.init_weights()
    def init_weights(self):
        initrange = 0.1
        nn.init.uniform_(self.input_emb.weight, -initrange, initrange)
        nn.init.zeros_(self.decoder.bias)
        nn.init.uniform_(self.decoder.weight, -initrange, initrange)
    def _generate_square_subsequent_mask(self, sz):
        return torch.log(torch.tril(torch.ones(sz, sz)))
    def forward(self, src):
        mask = self._generate_square_subsequent_mask(len(src)).to(src.device)
        src = self.input_emb(src) * math.sqrt(self.ninp)
        src = self.pos_encoder(src)
        output_enc = self.encoder(src, mask=mask)
        output_dec = self.decoder(output_enc)
        attention_maps = [layer.attn_weights for layer in self.encoder.layers]
        return F.log_softmax(output_dec, dim=-1), output_enc, attention_maps

## Step 5 — Generation, padding, batching

In [ ]:
# --- Autoregressive generation + padding/batching ---------------------------
def generate(model, prompts, new_tokens=6):
    input_tensor = prompts.to(device)
    for _ in range(new_tokens):
        output, _, _ = model(input_tensor)
        last = output[-1, :, :]
        token = torch.argmax(last, -1).view((1, -1))
        input_tensor = torch.cat((input_tensor, token), 0)
    return input_tensor

def pad(token_list, type_list="prompts"):
    max_length = max(len(x) for x in token_list)
    out = []
    for x in token_list:
        if type_list == "prompts":            # left-pad prompts
            out.append([tokenizer.token_to_id[pad_token]] * (max_length - len(x)) + x)
        else:                                  # answers: sequence + EOS + right-pad
            out.append(x + [tokenizer.token_to_id[eos_token]] +
                       [tokenizer.token_to_id[pad_token]] * (max_length - len(x)))
    return out, max_length

# REVERSE controls whether the TARGET answer is reversed before tokenising.
def get_batch(data, i, batch_size):
    j_end = min(i + batch_size, len(data))
    prompts = [tokenizer.encode(data[j][0]) for j in range(i, j_end)]
    padded_prompts, length_prompts = pad(prompts, "prompts")
    answers = [tokenizer.encode(to_reverse(data[j][1]) if REVERSE else data[j][1])
               for j in range(i, j_end)]
    padded_answers, length_answers = pad(answers, "answers")
    X = torch.stack([torch.tensor(x) for x in padded_prompts], 1)
    Y = torch.stack([torch.tensor(x) for x in padded_answers], 1)
    return X, Y, length_prompts, length_answers

## Step 6 — Accuracy metrics

In [ ]:
# --- Token-level accuracy (digit + full-sequence), from the workshop --------
def evaluate(input_data, batch_size=200):
    model.eval()
    correct_digit = total_digit = correct = total = 0.
    with torch.no_grad():
        for i in range(0, len(input_data), batch_size):
            prompts, target_answers, length_prompts, length_answers = get_batch(input_data, i, batch_size)
            prompts = prompts.to(device); target_answers = target_answers.to(device)
            output = generate(model, prompts, length_answers + 1)
            answers_tokens = output[length_prompts:, :]
            target_mask_pad = target_answers != tokenizer.token_to_id[pad_token]
            equality_test = answers_tokens == target_answers
            correct_digit += torch.logical_and(target_mask_pad, equality_test).sum().item()
            total_digit += target_mask_pad.sum().item()
            correct += torch.all(torch.logical_or(~target_mask_pad, equality_test), axis=0).float().sum().item()
            total += target_answers.shape[-1]
    return correct_digit / total_digit, correct / total

# --- Numeric prediction over a dataset (used for Task 3 numeric accuracy) ---
def predict_numeric(input_data, batch_size=200, new_tokens=6):
    """Return (prompts, true_ints, pred_ints). pred is None if unparseable."""
    model.eval()
    prompts_out, trues, preds = [], [], []
    with torch.no_grad():
        for i in range(0, len(input_data), batch_size):
            chunk = input_data[i:i + batch_size]
            enc = [tokenizer.encode(p) for p, _ in chunk]
            padded, Lp = pad(enc, "prompts")
            X = torch.stack([torch.tensor(x) for x in padded], 1).to(device)
            out = generate(model, X, new_tokens)
            gen = out[Lp:, :]
            for b, (p, t) in enumerate(chunk):
                s = tokenizer.decode(gen[:, b].tolist())
                s = s.split(eos_token)[0].replace(pad_token, "")
                prompts_out.append(p); trues.append(int(t))
                preds.append(parse_answer(s, reverse=REVERSE))
    return prompts_out, trues, preds

## Step 7 — Train the Forward model

Next-token prediction with cross-entropy, AdamW, causal mask — identical objective to the workshop. Training accuracy is monitored on a subset for speed; validation on the full set checks convergence.

In [ ]:
learning_rate = 1e-3
epochs = 30
batch_size = 100
# dropout=0.1 (the workshop's encoder-layer value) instead of the 0.5 default so the
# harder add+sub task converges; a cosine LR schedule anneals the rate over training,
# which is what lets the Reverse model escape its early plateau.
model = TransformerModel(ntoken=ntokens, ninp=128, nhead=16, nhid=64, nlayers=6, dropout=0.1)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

def train_epoch():
    model.train()
    total_loss, n_batch = 0., 0
    for i in tqdm(range(0, len(data_train) - 1, batch_size),
                  total=len(range(0, len(data_train) - 1, batch_size)), colour="GREEN"):
        n_batch += 1
        prompts, target_answers, length_prompts, length_answers = get_batch(data_train, i, batch_size)
        prompts = prompts.to(device); target_answers = target_answers.to(device)
        input_tensor = torch.cat((prompts, target_answers), 0)
        optimizer.zero_grad()
        output, _, _ = model(input_tensor)
        output_answers = output[length_prompts - 1:-1, :, :].reshape(-1, ntokens)
        loss = F.cross_entropy(output_answers, target_answers.view(-1))
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    return total_loss / n_batch

In [ ]:
# Training loop with per-epoch validation. Kept in the training notebook only;
# main_report.ipynb does NOT train (it loads the saved weights).
history = {"val_digit": [], "val_seq": [], "train_seq": [], "loss": []}
start = time.time()
for epoch in range(1, epochs + 1):
    t0 = time.time()
    loss = train_epoch()
    scheduler.step()
    val_digit, val_seq = evaluate(data_val)
    _, train_seq = evaluate(data_train[:5000])          # subset for speed
    history["val_digit"].append(val_digit); history["val_seq"].append(val_seq)
    history["train_seq"].append(train_seq); history["loss"].append(loss)
    print(f"epoch {epoch:2d} ({time.time()-t0:5.1f}s)  loss {loss:.3f} "
          f"| val digit {val_digit:.3f}  val seq {val_seq:.3f}  train seq {train_seq:.3f}")
print(f"total training time: {(time.time()-start)/60:.1f} min")

In [ ]:
# Save the trained weights for main_report.ipynb
torch.save(model.state_dict(), "LLMForward.pth")
print("saved LLMForward.pth")

### Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2), dpi=120)
ax.plot(history["train_seq"], "--", label="train seq acc")
ax.plot(history["val_seq"], label="val seq acc")
ax.axhline(0.78, color="red", ls=":", lw=1, label="0.78 target")
ax.set_xlabel("epoch"); ax.set_ylabel("sequence accuracy"); ax.set_ylim(0, 1)
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Quick sanity check

In [ ]:
model.eval()
print(f"{'prompt':>12} | {'true':>6} | {'pred':>6}")
for p, t in data_test[:12]:
    _, _, preds = (lambda d: predict_numeric(d))([(p, t)])
    print(f"{p:>12} | {t:>6} | {str(preds[0]):>6}")
val_digit, val_seq = evaluate(data_test)
print(f"\nHeld-out test: digit acc {val_digit:.3f} | sequence acc {val_seq:.3f}")